In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
# ============================================================
# DLGenAI Milestone-2 — full script
# ============================================================

!pip install -q sentence-transformers

from datasets import load_dataset
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

TRAIN_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'

# datasets-library version (needed for Q1, Q4)
train_ds = load_dataset('csv', data_files=TRAIN_PATH)['train']

# pandas version (easier for Q10-Q13)
df = pd.read_csv(TRAIN_PATH)
print("Columns:", df.columns.tolist())
print("Rows:", len(df))

option_cols = ['A', 'B', 'C', 'D', 'E']  # adjust if your dataset differs


# ---------------- Q1: combined_text length at index 51 ----------------
def add_combined(example):
    example['combined_text'] = example['prompt'] + ' ' + example['A']
    return example

train_ds = train_ds.map(add_combined)
print("\nQ1 combined_text length @ idx 51:", len(train_ds[51]['combined_text']))


# ---------------- Q2 & Q3: tokenizer vocab size / [SEP] id ----------------
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print("Q2 vocab_size:", tokenizer.vocab_size)
print("Q3 [SEP] id:", tokenizer.sep_token_id)


# ---------------- Q4: shape of tokenized prompt column ----------------
prompts = train_ds['prompt']
encoded = tokenizer(
    prompts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)
print("Q4 input_ids shape:", encoded['input_ids'].shape)


# ---------------- Q5: attention head dimensionality ----------------
hidden_size = 768
num_heads = 12
head_dim = hidden_size // num_heads
print("Q5 head_dim:", head_dim)


# ---------------- Q6 & Q7: last_hidden_state shape + CLS sum ----------------
model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

prompt0 = train_ds[0]['prompt']
inputs = tokenizer(prompt0, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

print("Q6 last_hidden_state shape:", outputs.last_hidden_state.shape)

cls_vec = outputs.last_hidden_state[0, 0, :]
print("Q7 CLS first-5 sum:", round(cls_vec[:5].sum().item(), 4))


# ---------------- Q8: attention weight to "fusion" ----------------
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
attn_inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    attn_out = model_attn(**attn_inputs)

tokens = tokenizer.convert_ids_to_tokens(attn_inputs['input_ids'][0])
print("Tokens:", tokens)
fusion_idx = tokens.index('fusion')

last_layer_attn = attn_out.attentions[-1]  # (batch, heads, seq, seq)
weight = last_layer_attn[0, 0, 0, fusion_idx].item()
print("Q8 attention weight [CLS] -> fusion:", round(weight, 4))


# ---------------- Q9: MiniLM cosine similarity ----------------
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

optionB0 = train_ds[0]['B']
emb1 = st_model.encode(prompt0, convert_to_tensor=True)
emb2 = st_model.encode(optionB0, convert_to_tensor=True)

sim = util.cos_sim(emb1, emb2)
print("Q9 cos_sim(prompt0, B0):", round(sim.item(), 4))


# ---------------- Q10: MAP@3 — TF-IDF vs MiniLM ----------------
tfidf_preds = []
for _, row in df.iterrows():
    texts = [row['prompt']] + [row[c] for c in option_cols]
    vec = TfidfVectorizer().fit_transform(texts)
    sims = cosine_similarity(vec[0:1], vec[1:]).flatten()
    ranked = sorted(zip(option_cols, sims), key=lambda x: -x[1])
    tfidf_preds.append([opt for opt, _ in ranked[:3]])

def map3(preds_list, answers):
    scores = []
    for preds, ans in zip(preds_list, answers):
        scores.append(1.0 / (preds.index(ans) + 1) if ans in preds else 0.0)
    return np.mean(scores)

tfidf_map3 = map3(tfidf_preds, df['answer'])
print("\nQ10 TF-IDF MAP@3:", round(tfidf_map3, 4))

prompt_embs = st_model.encode(df['prompt'].tolist(), convert_to_tensor=True)
option_embs = {c: st_model.encode(df[c].tolist(), convert_to_tensor=True) for c in option_cols}

minilm_preds = []
for i in range(len(df)):
    scores = {c: util.cos_sim(prompt_embs[i], option_embs[c][i]).item() for c in option_cols}
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    minilm_preds.append([opt for opt, _ in ranked[:3]])

minilm_map3 = map3(minilm_preds, df['answer'])
print("Q10 MiniLM MAP@3:", round(minilm_map3, 4))

count = sum(
    1 for t_preds, m_preds, ans in zip(tfidf_preds, minilm_preds, df['answer'])
    if ans not in t_preds and ans in m_preds
)
print("Q10 TF-IDF missed but MiniLM caught:", count)


# ---------------- Q11 & Q12: zero-shot classification ----------------
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row1 = df.iloc[1]
candidate_labels = [row1['A'], row1['B'], row1['C']]

result = classifier(row1['prompt'], candidate_labels)
print("\nQ11 top softmax score:", round(result['scores'][0], 4))

result_multi = classifier(row1['prompt'], candidate_labels, multi_label=True)
sum_softmax = sum(result['scores'])
sum_sigmoid = sum(result_multi['scores'])
print("Q12 |sum_softmax - sum_sigmoid|:", round(abs(sum_softmax - sum_sigmoid), 4))


# ---------------- Q13: flan-t5-small generative QA ----------------
gen_pipe = pipeline("text2text-generation", model="google/flan-t5-small")

row0 = df.iloc[0]
prompt_str = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    f"Answer with just the letter A or B."
)

gen_output = gen_pipe(prompt_str, max_new_tokens=5)
print("\nQ13 generated output:", gen_output[0]['generated_text'])

Generating train split: 0 examples [00:00, ? examples/s]

Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Rows: 2000


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


Q1 combined_text length @ idx 51: 614


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q2 vocab_size: 30522
Q3 [SEP] id: 102


ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).